# 기계론적 해석성 도구 - 가중치에서 직접 읽기

- Tutorial ID: `tut-9-1`
- Tutorial: 기계론적 해석성 도구
- Section ID: `s9-1-1`
- Section: 가중치에서 직접 읽기


In [ ]:
# ============================================================
# 코드 읽는 법 — 가중치에서 직접 읽기
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("기계론적 해석성: 트랜스포머 역설계 도구")
print("=" * 60)

def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

vocab_size = 8
d_model = 6
d_head = 3
n_heads = 2

vocab = ['<BOS>', 'cat', 'dog', 'runs', 'fast', 'the', 'a', '<EOS>']

np.random.seed(99)

W_E = np.random.randn(vocab_size, d_model) * 0.4
W_U = np.random.randn(d_model, vocab_size) * 0.3

layers_W_Q = [[np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads)] for _ in range(2)]
layers_W_K = [[np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads)] for _ in range(2)]
layers_W_V = [[np.random.randn(d_model, d_head) * 0.3 for _ in range(n_heads)] for _ in range(2)]
layers_W_O = [[np.random.randn(d_head, d_model) * 0.3 for _ in range(n_heads)] for _ in range(2)]

In [ ]:
# 도구 1: 바이그램 테이블 직접 추출

In [ ]:
print("
도구 1: 바이그램 테이블 (0-레이어 기여)")
print("-" * 45)

bigram_table = W_E @ W_U
bigram_probs = softmax(bigram_table)

print("상위 바이그램 예측 (확률 > 0.15):")
for i, src in enumerate(vocab):
        for j, dst in enumerate(vocab):
        if bigram_probs[i, j] > 0.15:
            print(f"  P({dst}|{src}) = {bigram_probs[i,j]:.3f}")

In [ ]:
# 도구 2: OV 회로 분석

In [ ]:
print("
도구 2: OV 회로 분석 (각 헤드)")
print("-" * 45)

for l in range(2):
        for h in range(n_heads):
                W_OV = layers_W_V[l][h] @ layers_W_O[l][h]
        C_OV = W_E @ W_OV @ W_U
        
        # 고유값 분석 (복사 동작 감지)
        eigenvalues = np.linalg.eigvals(C_OV)
        real_eig = np.real(eigenvalues)
        pos_eig_frac = np.sum(real_eig > 0) / len(real_eig)
        
        # 대각선 분석 (자기 복사)
        diag_mean = np.mean(np.diag(C_OV))
        off_diag_mean = (np.sum(C_OV) - np.sum(np.diag(C_OV))) / (vocab_size**2 - vocab_size)
        
        print(f"  레이어{l} 헤드{h}:")
        print(f"    양의 고유값 비율: {pos_eig_frac:.2f} "
              f"({'복사형' if pos_eig_frac > 0.6 else '비복사형'})")
        print(f"    대각 평균: {diag_mean:.3f}, 비대각 평균: {off_diag_mean:.3f}")
        if diag_mean > off_diag_mean:
            print(f"    → 자기 복사(self-copying) 동작 감지!")

In [ ]:
# 도구 3: QK 회로 분석 (어떤 토큰 쌍이 서로 주목하는가)

In [ ]:
print("
도구 3: QK 회로 분석")
print("-" * 45)

for l in range(2):
        for h in range(n_heads):
                W_QK = layers_W_Q[l][h] @ layers_W_K[l][h].T
        C_QK = W_E @ W_QK @ W_E.T
                C_QK_probs = softmax(C_QK)
        
        # 각 쿼리 토큰에 대해 가장 높은 키 찾기
        print(f"  레이어{l} 헤드{h} (쿼리→주목하는 키):")
                for i, src in enumerate(vocab[:4]):  # 처음 4개만
            top_key = vocab[np.argmax(C_QK_probs[i])]
            score = np.max(C_QK_probs[i])
            print(f"    '{src}' → '{top_key}' (점수={score:.3f})")

In [ ]:
# 도구 4: 합성 강도 행렬

In [ ]:
print("
도구 4: 레이어 간 합성 강도")
print("-" * 45)

def composition_score(W_A, W_B):
    """프로베니우스 노름 기반 합성 강도"""
    AB = W_A @ W_B
    return np.linalg.norm(AB) / (np.linalg.norm(W_A) * np.linalg.norm(W_B) + 1e-8)

print("K-합성 강도 (레이어1 헤드 × 레이어0 헤드):")
print("           L0H0    L0H1")
for h1 in range(n_heads):
        W_QK_1 = layers_W_Q[1][h1] @ layers_W_K[1][h1].T
        scores = []
        for h0 in range(n_heads):
                W_OV_0 = layers_W_V[0][h0] @ layers_W_O[0][h0]
        s = composition_score(W_QK_1.T, W_OV_0)
        scores.append(s)
    row = "  ".join(f"{s:.4f}" for s in scores)
    print(f"  L1H{h1}:   {row}")

expected = 1.0 / np.sqrt(d_head)
print(f"
  랜덤 기대값 ≈ 1/√{d_head} = {expected:.4f}")
print(f"  이보다 큰 값 → 실제 합성 발생!")

In [ ]:
# 도구 5: 항 중요도 분석 (Term Importance Analysis)

In [ ]:
print("
도구 5: 항 중요도 분석")
print("-" * 45)
print("""
각 '경로'가 최종 로짓에 얼마나 기여하는지 측정합니다.

방법:
1. 각 경로의 기여 텐서 계산: T_path = W_U · W_OV^h · A^h · W_E
2. 프로베니우스 노름으로 크기 측정: ||T_path||_F
3. 정규화하여 상대적 중요도 계산

이것으로 어떤 어텐션 헤드가 가장 중요한지 알 수 있습니다.

2-레이어 모델 분석 결과 (논문):
  - 직접 경로: 중간 중요도 (바이그램 기여)
  - 레이어0 헤드: 낮은 중요도 (주로 레이어1 헤드 지원)
  - 레이어1 인덕션 헤드: 높은 중요도!
  - V-합성 경로: 낮은 중요도 (무시 가능)
  
이것이 모델의 '핵심 메커니즘'을 식별하는 방법입니다.
""")

print("
✅ 기계론적 해석성 도구 구현 완료!")
print("이 도구들로 임의의 트랜스포머를 역설계할 수 있습니다.")